In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models

from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [3]:
batch_size = 32
learning_rate = 0.001
epochs = 5

image_size = 224

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

In [5]:
train_dataset = datasets.ImageFolder(
    root="../../datasets/cats_dogs/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root="../../datasets/cats_dogs/val",
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    root="../../datasets/cats_dogs/test",
    transform=test_transform
)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [7]:
print(train_dataset.classes)

num_classes = len(train_dataset.classes)

print(num_classes)

['Cat', 'Dog']
2


In [8]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

In [9]:
for param in model.parameters():
    param.requires_grad = False

In [10]:
in_features = model.fc.in_features

model.fc = nn.Linear(
    in_features,
    num_classes
)

In [11]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.fc.parameters(),
    lr=learning_rate
)

In [13]:
for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss:.4f} "
        f"Accuracy: {train_accuracy:.2f}%"
    )

KeyboardInterrupt: 

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

val_accuracy = 100 * correct / total

print("Validation Accuracy:", val_accuracy)

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print("Test Accuracy:", test_accuracy)

In [ ]:
torch.save(
    model.state_dict(),
    "models/transfer_learning_resnet18.pth"
)

print("Model Saved")

In [ ]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model.load_state_dict(
    torch.load(
        "models/transfer_learning_resnet18.pth",
        map_location=device
    )
)

model.to(device)

model.eval()